#### Multi Modal AI Assistant with Image Generation and Audio Synthesis

In [1]:
import os
import json
from openai import OpenAI
from dotenv import load_dotenv
import sqlite3
import gradio as gr

import base64               # to decode the image data
from io import BytesIO      # to wrap the image data into in-memory like file object
from PIL import Image       # to open the wrapped image file

In [2]:
load_dotenv(override=True)

GROQ_BASE_URL = os.getenv('GROQ_BASE_URL')
GROQ_API_KEY = os.getenv('GROQ_API_KEY')

GEMINI_BASE_URL = os.getenv('GEMINI_BASE_URL')
GEMINI_API_KEY = os.getenv('GEMINI_API_KEY')

openai = OpenAI()

groq = OpenAI(base_url = GROQ_BASE_URL, api_key = GROQ_API_KEY)

gemini = OpenAI(base_url = GEMINI_BASE_URL, api_key = GEMINI_API_KEY)


In [3]:
# database file path

DB = '/Users/dorababulalam/GitHub/gen-ai/llm_engineering_practice/002_week/day_4/prices.db'

In [4]:
# database tool

def get_ticket_price(city):
    print(f'Database Tool called for {city}')
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute('SELECT price FROM prices WHERE city = ?',(city.lower(),))
        result = cursor.fetchone()
        return f'Return ticket price for the {city} is ${result[0]}' if result else 'No price details found for this city'

In [5]:
my_tool_schema = {
    'name' : 'get_ticket_price',
    'description' : 'Get the return ticket price for the given destination city',
    'parameters':{
        'type':'object',
        'properties':{
            'destination_city':{
                'type':'string',
                'description':'The city that the customer wants to travel to'
            },
        },
        'required':['destination_city'],
        'additionalProperties':False
    }
}

In [6]:
tools = [{'type':'function', 'function':my_tool_schema}]        # indicates the list of tools. Each {} indicate 1 tool
# here the type indicates it is a function and function indicates the schema of that tool
tools

[{'type': 'function',
  'function': {'name': 'get_ticket_price',
   'description': 'Get the return ticket price for the given destination city',
   'parameters': {'type': 'object',
    'properties': {'destination_city': {'type': 'string',
      'description': 'The city that the customer wants to travel to'}},
    'required': ['destination_city'],
    'additionalProperties': False}}}]

In [7]:
get_ticket_price('london')

Database Tool called for london


'Return ticket price for the london is $799.0'

In [ ]:
# image tool

def artist(city):
    try:
        image_response = openai.images.generate(
            model = 'gpt-image-1-mini',
            prompt = f'An image representing a vacation in {city}, showing tourist spots and eveything unique about the {city}, in a vibrant pop-up style',
            size = '1024x1024',
            n = 1
        )

        image_base64 = image_response.data[0].b64_json
        image_data = base64.b64decode(image_base64)
        return Image.open(BytesIO(image_data))
        
    except Exception as e:
        return f'Error occured in Image Tool : {e}'

In [ ]:
# speech tool

def talker(message):
    try:
        response = openai.audio.speech.create(
            model = 'gpt-4o-mini-tts',
            voice = 'onyx',
            input = message
        )
        return response.content
        
    except Exception as e:
        return f'Error occured in Audio Tool : {e}'

In [10]:
system_message = """
You are a helpful assistant for an Airline called FlightAI.
Give short, courteous answers, no more than 1 sentence.
Always be accurate. If you don't know the answer, say so.
"""

In [ ]:
def chatbot(history):
    try:
        history = [{'role':h['role'], 'content':h['content']} for h in history]
        messages = [{'role':'system', 'content':system_message}] + history
        response = groq.chat.completions.create(
            model = 'openai/gpt-oss-20b',
            messages = messages,
            tools = tools
        )
        cities = []
        image = None

        while response.choices[0].finish_reason == 'tool_calls':
            message = response.choices[0].message
            responses, cities = handle_tool_calls_and_return_cities(message)
            messages.append(message)
            messages.extend(responses)
            response = groq.chat.completions.create(
                model = 'openai/gpt-oss-20b',
                messages = messages,
                tools = tools
            )
        
        reply = response.choices[0].message.content
        history += [{'role':'assistant', 'content':reply}]

        voice = talker(reply)

        if cities:
            image = artist(cities[0])
        
        return history, voice, image
        
    except Exception as e:
        return f'Error occured in Chatbot : {e}'

In [12]:
def handle_tool_calls_and_return_cities(message):
    responses = []
    cities = []

    for tool_calls in message.tool_calls:
        if tool_call.function.name == 'get_ticket_price':
            arguments = json.loads(tool_call.function.arguments)
            city = arguments.get('destination_city')
            cities.append(city)
            price_details = get_ticket_price(city)
            responses.append({
                'role':'tool',
                'content':price_details,
                'tool_call_id':tool_call.id
            })
    return responses, cities


#### 3 Types of Gradio UI

- gr.Interface() is for standard, simple UIs

- gr.ChatInterface() is for standard Chatbot UIs

- gr.Blocks() is for custom UIs where **we can control the components and callbacks**

In [ ]:
def put_message_in_chatbot(message, history):
    return "", history + [{'role':'user', 'content':message}]

with gr.Blocks() as ui:

    with gr.Row():
        conversation = gr.Chatbot(height=500)
        image_output = gr.Image(height=500, interactive=False)

    with gr.Row():
        audio_output = gr.Audio(autoplay=True)

    with gr.Row():
        input_message = gr.Textbox(label='Chat with your AI Assistant')

    
    input_message.submit(
        put_message_in_chatbot, 
        inputs=[input_message, conversation], 
        outputs=[input_message, conversation]).then(
        chatbot, 
        inputs=conversation, 
        outputs=[conversation, audio_output, image_output]
    )

    # the above statements indicate that
    # when the user presses Enter or submits text:
    # put_message_in_chatbot runs and it clears the input_message textbox and adds the user message to conversation
    # then() calls chatbot, it passes the updated conversation
    
    ui.launch(inbrowser=True)

* Running on local URL:  http://127.0.0.1:7888
* To create a public link, set `share=True` in `launch()`.


Traceback (most recent call last):
  File "/Users/dorababulalam/GitHub/gen-ai/llm_engineering_practice/.venv/lib/python3.12/site-packages/gradio/queueing.py", line 870, in process_events
    response = await route_utils.call_process_api(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/dorababulalam/GitHub/gen-ai/llm_engineering_practice/.venv/lib/python3.12/site-packages/gradio/route_utils.py", line 408, in call_process_api
    output = await app.get_blocks().process_api(
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/dorababulalam/GitHub/gen-ai/llm_engineering_practice/.venv/lib/python3.12/site-packages/gradio/blocks.py", line 2330, in process_api
    data = await self.postprocess_data(
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/dorababulalam/GitHub/gen-ai/llm_engineering_practice/.venv/lib/python3.12/site-packages/gradio/blocks.py", line 2095, in postprocess_data
    await processing_utils.async_move_files_to_cache(
  File "/Users/doraba